
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Building Single Agents with LangChain

##Overview 

In this hands-on lab, you will build AI agents that leverage Unity Catalog (UC) functions as tools within the LangChain framework. You will create UC functions for analyzing NYC taxi trip data, integrate them with LangChain toolkits, and build an agent capable of reasoning and taking action using foundation models hosted in Mosaic AI Model Serving.

## Learning Objectives

By the end of this lab, you will be able to:
- Test pre-built Unity Catalog functions independently before agent integration
- Configure and integrate Unity Catalog functions with LangChain using the `UCFunctionToolkit`
- Build and execute a LangChain agent with tool-calling capabilities
- Analyze agent execution traces using MLflow for debugging and optimization

**Note**: Before starting this lab, it is highly recommended that you first complete the previous demonstration.

## A. Environment Setup and Prerequisites

### A1. Compute Requirements

**🚨 REQUIRED - SELECT SERVERLESS COMPUTE**

This course has been configured to run on Serverless compute. While classic compute may also work, testing has been performed on serverless.

**This demo was tested using version 5 of Serverless compute.** To ensure that you are using the correct version of Serverless, please [see this documentation on viewing and changing your notebook's Serverless version.](https://docs.databricks.com/aws/en/compute/serverless/dependencies) 

### A2. Install Dependencies

As part of the workspace setup, several Python libraries have been installed. To see the list of notebook-scoped libraries, please read [this documentation](https://docs.databricks.com/aws/en/compute/serverless/dependencies#configure-environment-for-job-tasks). 

**NOTE:** If you are familiar with `langchain-databricks`, note that `databricks-langchain` replaces it. This lab uses LangChain, but a similar approach can be applied to other libraries.

In [0]:
%run ../Includes/Classroom-Setup-2.2

### A3. Inspect NYC Taxi Dataset

The dataset that will be used for this lab will come from the table `samples.nyctaxi.trips`, which is a sample dataset that [is available by default on UC-enabled workspaces](https://docs.databricks.com/aws/en/discover/databricks-datasets). Run the next cell to query the first few rows of the dataset. 

In [0]:
df = spark.read.table('samples.nyctaxi.trips')
display(df.limit(5))

## B. Initialize the Databricks Function Client

The `DatabricksFunctionClient` provides a programmatic interface for executing Unity Catalog functions. We configure it for serverless execution mode to align with our compute requirements.

**TODO**: Initialize the client for serverless compute.

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

## Initialize the client for serverless compute
client = <FILL_IN>

##### Task B: Initialize the Databricks Function Client ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# Initialize the client for serverless compute
client = DatabricksFunctionClient(execution_mode="serverless")
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


from unitycatalog.ai.core.databricks import DatabricksFunctionClient

# Initialize the client for serverless compute
client = DatabricksFunctionClient(execution_mode="serverless")



## C. Inspect Functions Before Agent Integration
As a part of the classroom setup, 2 agent tools have been created for you:
1. `avg_fare_by_zip`: Calculates the average fare amount for trips from a specific pickup ZIP code. Returns the average fare as a numeric value.
1. `cnt_lng_dist_trip`: Counts the number of trips longer than the specified distance from a given pickup ZIP code. Returns the count as an integer.

### UI Inspection Instructions
1. **Navigate to the Catalog**
- In the left sidebar, click **Catalog**.

2. **Locate Your Workspace Catalog and Schema**
- Open the **dbacademy** catalog.
- Select the schema that begins with **labuser** (this is the schema you have been working in).

3. **View Functions**
- Click on **Functions** in the schema sidebar.
- Locate the functions `avg_fare_by_zip` and `cnt_lng_dist_trip`

4. **Inspect Function Details**
- Click on a function name to open its details.
- Review:
- **Comments on input parameters**
- **The referenced table or data source**
- **Additional metadata** associated with the function

### C1. Test Functions Before Agent Integration

Before integrating these functions into our agent, let's verify they work correctly by testing them directly with SQL queries.

In [0]:
%sql
-- TODO
-- -- Test the average fare function with ZIP code 10001
SELECT <FILL_IN> AS manhattan_avg_fare;

##### Task C1.1: Test Average Fare Function ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
-- Test the average fare function with ZIP code 10001
SELECT avg_fare_by_zip(10001) AS manhattan_avg_fare;
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:
%sql


-- Test the average fare function with ZIP code 10001
SELECT avg_fare_by_zip(10001) AS manhattan_avg_fare;



In [0]:
%sql
-- TODO
-- -- Test the long distance trips function with ZIP code 10001 and minimum distance of 10 miles
SELECT <FILL_IN> AS long_trips_count;

##### Task C1.2: Test Long Distance Trips Function ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
-- Test the long distance trips function with ZIP code 10001 and minimum distance of 10 miles
SELECT cnt_lng_dist_trip(10001, 10.0) AS long_trips_count;
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:
%sql


-- Test the long distance trips function with ZIP code 10001 and minimum distance of 10 miles
SELECT cnt_lng_dist_trip(10001, 10.0) AS long_trips_count;



## D. Integrate Unity Catalog Functions with LangChain

Now that we know the functions work with basic SQL, we'll leverage `databricks_langchain` to wrap our UC functions as tools that can be directly integrated into LangChain. The first step is to define the tool list, which we will do next.

### D1. Define the Tool List

**TODO:** Create a list of the functions we want to use and format them with the catalog and schema names (this is a requirement for our agent framework).

In [0]:
tool_list_raw = [
    <FILL_IN>,
    <FILL_IN>
]

function_names = []
for tool in tool_list_raw:
    tool = <FILL_IN> + '.' + <FILL_IN> + '.' + tool
    function_names.append(tool)

print(f"Tool list: {function_names}")

##### Task D1: Define the Tool List ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
tool_list_raw = [
'avg_fare_by_zip',
'cnt_lng_dist_trip'
]

function_names = []
for tool in tool_list_raw:
  tool = catalog_name + '.' + schema_name + '.' + tool
  function_names.append(tool)

print(f"Tool list: {function_names}")
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


tool_list_raw = [
'avg_fare_by_zip',
'cnt_lng_dist_trip'
]

function_names = []
for tool in tool_list_raw:
  tool = catalog_name + '.' + schema_name + '.' + tool
  function_names.append(tool)

print(f"Tool list: {function_names}")



### D2. Create the UCFunctionToolkit

**TODO:** Create the toolkit that wraps our Unity Catalog functions and makes them available as LangChain tools using `UCFunctionToolkit`.

In [0]:
from databricks_langchain import UCFunctionToolkit

## Create a toolkit with the Unity Catalog functions
toolkit = <FILL_IN>
tools = <FILL_IN>

##### Task D2: Create the UCFunctionToolkit ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
from databricks_langchain import UCFunctionToolkit

# Create a toolkit with the Unity Catalog functions
toolkit = UCFunctionToolkit(function_names=function_names)
tools = toolkit.tools
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


from databricks_langchain import UCFunctionToolkit

# Create a toolkit with the Unity Catalog functions
toolkit = UCFunctionToolkit(function_names=function_names)
tools = toolkit.tools



### D3. Test the Toolkit
Finally we are ready to test our toolkit. Note that the outputs from the following query will be the exact same as above since only the syntax for the query has changed from SQL to using `client.execute_function()`
> Recall that `client` is defined as `client = DatabricksFunctionClient(execution_mode="serverless")` from your work above.

**TODO:** Test the toolkit by executing sample payloads using the `DatabricksFunctionClient`.

In [0]:
## Test the first tool (average fare by pickup ZIP)
payload1 = <FILL_IN>
payload1_test_result = client.execute_function(
    function_name=<FILL_IN>,
    parameters=payload1
)
print(payload1_test_result.value)

##### Task D3.1: Test First Tool ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
# Test the first tool (average fare by pickup ZIP)
payload1 = {'pickup_zip_code': 10001}
payload1_test_result = client.execute_function(
function_name=tools[0].uc_function_name,
parameters=payload1
)
print(payload1_test_result.value)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


# Test the first tool (average fare by pickup ZIP)
payload1 = {'pickup_zip_code': 10001}
payload1_test_result = client.execute_function(
function_name=tools[0].uc_function_name,
parameters=payload1
)
print(payload1_test_result.value)



In [0]:
## Test the second tool (count long distance trips)
payload2 = <FILL_IN>
payload2_test_result = client.execute_function(
    function_name=<FILL_IN>,
    parameters=payload2
)
print(payload2_test_result.value)

##### Task D3.2: Test Second Tool ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
# Test the second tool (count long distance trips)
payload2 = {
'pickup_zip_code': 10001,
'min_distance': 10.0
}
payload2_test_result = client.execute_function(
  function_name=tools[1].uc_function_name,
  parameters=payload2
)
print(payload2_test_result.value)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


# Test the second tool (count long distance trips)
payload2 = {
'pickup_zip_code': 10001,
'min_distance': 10.0
}
payload2_test_result = client.execute_function(
  function_name=tools[1].uc_function_name,
  parameters=payload2
)
print(payload2_test_result.value)



## E. Configure and Execute the Agent

Now that we have our toolkit configured and tested, we're ready to create an agent configuration file that will be decoupled from the agent code. As a part of the classroom setup, a JSON file has been loaded into the same directory where this lab is located, called `lab_agent.json`.
1. Navigate to the left menu and inspect the file `lab_agent.json` before completing the next task.

### E1. Load Agent Configuration

**TODO:** Load the agent configuration from the JSON file and extract the necessary parameters.

In [0]:
import json

## Load JSON file
with open("./lab_agent.json", "r") as f:
    config = <FILL_IN>

llm_endpoint = <FILL_IN>
llm_temperature = <FILL_IN>
system_prompt = <FILL_IN>

print("Endpoint:", llm_endpoint)
print("Temperature:", llm_temperature)
print("System Prompt:", system_prompt)

##### Task E1: Load Agent Configuration ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
import json

# Load JSON file
with open("./lab_agent.json", "r") as f:
  config = json.load(f)

llm_endpoint = config['llm_endpoint']
llm_temperature = config['llm_temperature']
system_prompt = config["system_prompt"]

print("Endpoint:", llm_endpoint)
print("Temperature:", llm_temperature)
print("System Prompt:", system_prompt)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


import json

# Load JSON file
with open("./lab_agent.json", "r") as f:
  config = json.load(f)

llm_endpoint = config['llm_endpoint']
llm_temperature = config['llm_temperature']
system_prompt = config["system_prompt"]

print("Endpoint:", llm_endpoint)
print("Temperature:", llm_temperature)
print("System Prompt:", system_prompt)



### E2. Import Required Libraries and Initialize Components

**TODO:** Import the necessary libraries and initialize the language model using `ChatDatabricks`.

In [0]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import ChatPromptTemplate
from databricks_langchain import ChatDatabricks
import mlflow

## Initialize the language model using ChatDatabricks
llm_config = <FILL_IN>

##### Task E2: Initialize Language Model ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import ChatPromptTemplate
from databricks_langchain import ChatDatabricks
import mlflow

# Initialize the language model
llm_config = ChatDatabricks(
  endpoint=llm_endpoint,
  temperature=llm_temperature
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import ChatPromptTemplate
from databricks_langchain import ChatDatabricks
import mlflow

# Initialize the language model
llm_config = ChatDatabricks(
  endpoint=llm_endpoint,
  temperature=llm_temperature
)



### E3. Define the Prompt Template

**TODO:** Create the prompt template using the _system prompt_ and conversation structure.

In [0]:
prompt_payload = ChatPromptTemplate.from_messages(
    [
        (
            <FILL_IN>,
            <FILL_IN>,
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

##### Task E3: Define Prompt Template ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
prompt_payload = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      system_prompt,
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
  ]
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


prompt_payload = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      system_prompt,
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
  ]
)



### E4. Enable MLflow Tracing and Create Agent Configuration

**TODO:** Enable MLflow autologging and create the agent configuration using `create_tool_calling_agent` from the `langchain` library.

In [0]:
## Enable MLflow tracing
<FILL_IN>

## Create the agent configuration
agent_config = <FILL_IN>

##### Task E4: Enable MLflow and Create Agent ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
# Enable MLflow tracing
mlflow.langchain.autolog()

# Create the agent configuration
agent_config = create_tool_calling_agent(
  llm_config,
  tools,
  prompt_payload
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


# Enable MLflow tracing
mlflow.langchain.autolog()

# Create the agent configuration
agent_config = create_tool_calling_agent(
  llm_config,
  tools,
  prompt_payload
)



### E5. Execute the Agent

**TODO:** Create the agent executor and run it with a query about NYC taxi data use `AgentExecutor()` and get the response.

**Bonus**: Construct a query that will call both tools.

In [0]:
agent_executor = AgentExecutor(agent=<FILL_IN>, tools=<FILL_IN>, verbose=True)
response = agent_executor.invoke(
    {
        "input": <FILL_IN>
    }
)

##### Task E5: Execute the Agent ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
agent_executor = AgentExecutor(agent=agent_config, tools=tools, verbose=True)
response = agent_executor.invoke(
  {
    "input": "What's the average fare for trips from ZIP code 10001 and how many trips from that ZIP code are longer than 15 miles?"
  }
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


agent_executor = AgentExecutor(agent=agent_config, tools=tools, verbose=True)
response = agent_executor.invoke(
  {
    "input": "What's the average fare for trips from ZIP code 10001 and how many trips from that ZIP code are longer than 15 miles?"
  }
)



## F. Analyze Agent Response and Tracing

### F1. Parse the Agent's Response

**TODO:** Extract and display the agent's response in a readable format.

In [0]:
## Extract text segments from the response
output_segments = <FILL_IN>

print(output_segments)

##### Task F1: Parse Agent Response ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
# Extract text segments from the response
output_segments = response['output']

print(output_segments)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


# Extract text segments from the response
output_segments = response['output']

print(output_segments)



### F2. Experiment with Different Queries

**TODO:** Try running the agent with different queries to test its capabilities.

**Bonus**: Create a query that demonstrates the agent's ability to search among different input values like supplying more than one ZIP code.

In [0]:
## Try a different query
custom_response = agent_executor.invoke(
    {
        "input": <FILL_IN>
    }
)

##### Task F2: Experiment with Different Queries ANSWER
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
# Try a different query
custom_response = agent_executor.invoke(
  {
    "input": "Compare the average fare for ZIP codes 10001 and 10002, and tell me which one has more trips over 20 miles"
  }
)
<!-------------------END SOLUTION CODE------------------->
</code></pre>

<script>
function copyBlock() {
const el = document.getElementById("copy-block");
if (!el) return;

const text = el.innerText;

// Preferred modern API
if (navigator.clipboard && navigator.clipboard.writeText) {
navigator.clipboard.writeText(text)
.then(() => alert("Copied to clipboard"))
.catch(err => {
console.error("Clipboard write failed:", err);
fallbackCopy(text);
});
} else {
fallbackCopy(text);
}
}

function fallbackCopy(text) {
const textarea = document.createElement("textarea");
textarea.value = text;
textarea.style.position = "fixed";
textarea.style.left = "-9999px";
document.body.appendChild(textarea);
textarea.select();
try {
document.execCommand("copy");
alert("Copied to clipboard");
} catch (err) {
console.error("Fallback copy failed:", err);
alert("Could not copy to clipboard. Please copy manually.");
} finally {
document.body.removeChild(textarea);
}
}
</script>
</details>

In [0]:


# Try a different query
custom_response = agent_executor.invoke(
  {
    "input": "Compare the average fare for ZIP codes 10001 and 10002, and tell me which one has more trips over 20 miles"
  }
)



## G. Review and Analysis

### MLflow Trace Analysis

Review the MLflow Trace UI that appears in the outputs above. Click on the **Summary** tab to see:

- **Which tools were called**: Both tools should have been invoked for the queries
- **What parameters were passed**: Examine the inputs to each UC function
- **The results returned from each tool**: Review the numeric outputs
- **The final response generated by the agent**: See how the LLM synthesized the tool results

### Key Observations

Based on your agent execution, consider:

1. **Tool Selection**: Did the agent choose the appropriate tools for each query?
2. **Parameter Passing**: Were the correct parameters passed to each function?
3. **Response Quality**: How well did the agent synthesize the tool results into a coherent answer?
4. **Error Handling**: What happens if you provide invalid ZIP codes or parameters?

## Conclusion

You have successfully built a LangChain agent that leverages Unity Catalog functions as tools for analyzing NYC taxi trip data. This approach demonstrates how to build production-ready AI agents that can reason about and act upon your organization's data assets while maintaining the governance, security, and lineage tracking provided by Unity Catalog.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>